In [1]:
%pip install scikit-learn


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
%pip install sentence-transformers

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
import re
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

c:\Users\Hemanth Sai\OneDrive\Desktop\info\queytube\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
df = pd.read_csv("video_with_transcripts.csv")

print("Dataset Shape:", df.shape)

df.head()

Dataset Shape: (250, 7)


,video_id,title,publish_date,year,month,title_length,transcript
0,1Z-8HxlLAHk,Kids and young people: stay curious and be wil...,2026-03-30 12:17:27+00:00,2026,3,72,like and even at a young age like learning to ...
1,ehkDsePv3W8,Here's a cool and easy way to work with colors...,2026-03-29 12:43:24+00:00,2026,3,72,Working with colors in 3JS. One cool way to wo...
2,uWRdzJTpcpI,Do web devs NEED to understand low-level progr...,2026-03-28 12:32:40+00:00,2026,3,88,A lot of what the curriculum was trying to do ...
3,GC5CLCgnvm0,"When things are new and a little scary, embrac...",2026-03-27 12:18:22+00:00,2026,3,79,what this future's going to look like is not g...
4,tZef2ZzbyuQ,What happens when the model CAN'T fix it? Inte...,2026-03-27 10:00:57+00:00,2026,3,99,Welcome back to the Free Code Camp podcast. I'...


In [5]:
required_cols = ["video_id", "title", "publish_date", "transcript"]

for col in required_cols:
    if col not in df.columns:
        print(f"{col} column missing ❌")
    else:
        print(f"{col} column exists ✅")

video_id column exists ✅
title column exists ✅
publish_date column exists ✅
transcript column exists ✅


In [6]:
def clean_text(text):
    
    if pd.isna(text):
        return text
    
    text = str(text)

    # remove newline and tab
    text = re.sub(r'\n|\t', ' ', text)

    # remove special characters
    text = re.sub(r'[^\w\s]', '', text)

    # remove multiple spaces
    text = re.sub(r'\s+', ' ', text)

    return text.strip()

In [7]:
df["title"] = df["title"].apply(clean_text)

df["transcript"] = df["transcript"].apply(clean_text)

df.head()

,video_id,title,publish_date,year,month,title_length,transcript
0,1Z-8HxlLAHk,Kids and young people stay curious and be will...,2026-03-30 12:17:27+00:00,2026,3,72,like and even at a young age like learning to ...
1,ehkDsePv3W8,Heres a cool and easy way to work with colors ...,2026-03-29 12:43:24+00:00,2026,3,72,Working with colors in 3JS One cool way to wor...
2,uWRdzJTpcpI,Do web devs NEED to understand lowlevel progra...,2026-03-28 12:32:40+00:00,2026,3,88,A lot of what the curriculum was trying to do ...
3,GC5CLCgnvm0,When things are new and a little scary embraci...,2026-03-27 12:18:22+00:00,2026,3,79,what this futures going to look like is not go...
4,tZef2ZzbyuQ,What happens when the model CANT fix it Interv...,2026-03-27 10:00:57+00:00,2026,3,99,Welcome back to the Free Code Camp podcast Im ...


In [8]:
print("Null transcripts:", df["transcript"].isnull().sum())

Null transcripts: 6


In [9]:
df = df.dropna(subset=["transcript"])

In [10]:
df = df[df["transcript"].str.strip() != ""]

In [11]:
df = df.rename(columns={"publish_date": "datetime"})

In [12]:
df["datetime"] = pd.to_datetime(df["datetime"])

In [13]:
df = df[["video_id", "title", "datetime", "transcript"]]

In [14]:
df.head()

,video_id,title,datetime,transcript
0,1Z-8HxlLAHk,Kids and young people stay curious and be will...,2026-03-30 12:17:27+00:00,like and even at a young age like learning to ...
1,ehkDsePv3W8,Heres a cool and easy way to work with colors ...,2026-03-29 12:43:24+00:00,Working with colors in 3JS One cool way to wor...
2,uWRdzJTpcpI,Do web devs NEED to understand lowlevel progra...,2026-03-28 12:32:40+00:00,A lot of what the curriculum was trying to do ...
3,GC5CLCgnvm0,When things are new and a little scary embraci...,2026-03-27 12:18:22+00:00,what this futures going to look like is not go...
4,tZef2ZzbyuQ,What happens when the model CANT fix it Interv...,2026-03-27 10:00:57+00:00,Welcome back to the Free Code Camp podcast Im ...


In [15]:
print("Final Dataset Shape:", df.shape)

print("\nNull values:\n", df.isnull().sum())

Final Dataset Shape: (244, 4)

Null values:
 video_id      0
title         0
datetime      0
transcript    0
dtype: int64


In [16]:
df.to_csv("cleaned_transcripts.csv", index=False)

print("Dataset saved as cleaned_transcripts.csv")

Dataset saved as cleaned_transcripts.csv


In [17]:
df = pd.read_csv("cleaned_transcripts.csv")

print("Dataset Shape:", df.shape)

df.head()

Dataset Shape: (244, 4)


,video_id,title,datetime,transcript
0,1Z-8HxlLAHk,Kids and young people stay curious and be will...,2026-03-30 12:17:27+00:00,like and even at a young age like learning to ...
1,ehkDsePv3W8,Heres a cool and easy way to work with colors ...,2026-03-29 12:43:24+00:00,Working with colors in 3JS One cool way to wor...
2,uWRdzJTpcpI,Do web devs NEED to understand lowlevel progra...,2026-03-28 12:32:40+00:00,A lot of what the curriculum was trying to do ...
3,GC5CLCgnvm0,When things are new and a little scary embraci...,2026-03-27 12:18:22+00:00,what this futures going to look like is not go...
4,tZef2ZzbyuQ,What happens when the model CANT fix it Interv...,2026-03-27 10:00:57+00:00,Welcome back to the Free Code Camp podcast Im ...


In [21]:
with open("queries.txt", "r") as f:
    queries = [line.strip() for line in f.readlines()]

print("Total queries:", len(queries))

queries[:10]

Total queries: 75


['How to get started with coding for beginners',
 'Best way to learn programming in 2025',
 'What is the future of web development',
 'How to use Three.js for creative coding',
 'Beginner guide to JavaScript frameworks',
 'How to build projects with React step by step',
 'What skills do software developers need today',
 'How to learn full stack development efficiently',
 'Tips for mastering Python programming',
 'What is the best programming language to learn first']

In [22]:
print("Loading search queries...\n")

queries_df = pd.read_csv("queries.csv")
queries = queries_df["query"].tolist()

print("Total queries loaded:", len(queries))
print("Sample queries:", queries[:5], "\n")

Loading search queries...

Total queries loaded: 75
Sample queries: ['How to get started with coding for beginners', 'Best way to learn programming in 2025', 'What is the future of web development', 'How to use Three.js for creative coding', 'Beginner guide to JavaScript frameworks'] 



In [23]:
print("Mapping queries to videos...\n")

mapping = []

stopwords = {
    "how", "what", "is", "the", "a", "an", "to", "can", "i", "in", "of", "for",
    "do", "does", "did", "you", "your", "with", "on", "and", "it", "this",
    "learn", "english", "scene", "dialogue", "conversation", "movie",
    "series", "tv"
}

for query in queries:
    q = query.lower()

    # remove punctuation
    q = re.sub(r"[^\w\s]", "", q)

    words = [w for w in q.split() if w not in stopwords]

    best_match = None
    best_count = 0

    for _, row in df.iterrows():
        text = (str(row["title"]) + " " + str(row["transcript"])).lower()

        count = sum(word in text for word in words)

        if count > best_count:
            best_count = count
            best_match = row["video_id"]

    video_id = best_match
    mapping.append((query, video_id))

Mapping queries to videos...



In [24]:
mapping_df = pd.DataFrame(mapping, columns=["query", "relevant_video_id"])

mapping_df.to_csv("query_video_map.csv", index=False)

print("Query -> Video mapping completed\n")

print("\nSample mappings:\n")
print(mapping_df.head())

Query -> Video mapping completed


Sample mappings:

                                          query relevant_video_id
0  How to get started with coding for beginners       IRcL5as-RaE
1         Best way to learn programming in 2025       uJrh9GHrC38
2         What is the future of web development       tZef2ZzbyuQ
3       How to use Three.js for creative coding       tdsQwuyS6DQ
4       Beginner guide to JavaScript frameworks       jydYq7oAtD8
